# Limits of direct methods

In [ ]:
#    APM41012EP course notebook - Chapter 5 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Iterative methods for solving linear systems
#    Poisson 1D, 2D, 3D
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import time
import numpy as np

from scipy.sparse import diags
from scipy.sparse.linalg import spsolve, cg, norm

## Poisson equation

We want to solve the elliptic problem given by the Poisson equation subject to Dirichlet boundary conditions:

$$
\left\{
\begin{aligned}
-\Delta u & =  f & \text{in} \; \Omega  \\
        u & =  g & \text{on}  \;  \partial \Omega
\end{aligned}
\right.
$$

In [ ]:
def eig_val_1d(nx, dx, i):
    a = (4/(dx**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2

def eig_val_2d(nx, dx, ny, dy, i, j):
    a = (4/(dx**2))
    b = (4/(dy**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2 + b*np.sin((np.pi*j)/(2*(ny+1)))**2

def eig_val_3d(nx, dx, ny, dy, nz, dz, i, j, k):
    a = (4/(dx**2))
    b = (4/(dy**2))
    c = (4/(dz**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2 + b*np.sin((np.pi*j)/(2*(ny+1)))**2 + c*np.sin((np.pi*k)/(2*(nz+1)))**2

## 1d case

$$
\left\{
\begin{aligned}
- & u''(x)  =  f(x) \quad \text{in} \; \Omega = [0,1] \quad \text{with} \; f(x)=1\\
  & u(0)  =  0 \; \text{and} \; u(1) = 0
\end{aligned}
\right.
$$

In [ ]:
nx = 122500
nx = 122500
dx = 1/(nx+1)

# building the sparse matrix
diag = np.repeat(2/dx**2, nx)
diag_x = np.repeat(-1/dx**2, nx-1)
a  = diags([diag, diag_x, diag_x], [0, -1, 1])

# right-hand side
b = np.ones(nx)

print(f"1d case: nx = {nx}")
print(f"Matrix size: ({nx} x {nx})")
print(f"Bandwidth: {3}")
print(f"Entries of the factorised matrix: {nx*3:,}".replace(',',' '))
print(f"Condition number: {eig_val_1d(nx, dx, nx)/eig_val_1d(nx, dx, 1)}")
norm_a = eig_val_1d(nx, dx, nx)
print(f"Norm of A: {norm_a}")

t1 = time.time()    
ulu = spsolve(a.tocsr(), b)
t2 = time.time()
print("\nSolution using a direct method")
print(f"Execution time (s): {t2-t1}")
res = np.linalg.norm(b - a.dot(ulu))
print(f"||Ax - b|| / ||A|| ||x|| + ||b|| = {res / (norm_a * np.linalg.norm(ulu) + np.linalg.norm(b))}")

## 2d case

$$
\left\{
\begin{aligned}
-\Delta u(x,y) & =  f(x,y) \quad \text{ in } \; \Omega = [0,1] \times [0,1]  \quad \text{with} \; f(x)=1 \\
        u(x,y) & =  0 \quad \text{ on }  \;  \partial \Omega
\end{aligned}
\right.
$$

In [ ]:
nx = 350
ny = 350
dx = 1/(nx+1)
dy = 1/(ny+1)

# building the sparse matrix
diag = np.repeat(2/dx**2 + 2/dy**2, nx*ny)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny)
diag_y = np.repeat(-1/dy**2, nx*(ny-1))
a = diags([diag, diag_x, diag_x, diag_y, diag_y], [0, -1, 1, -nx, nx])

# right-hand side
b = np.ones(nx*ny)

print(f"2d case: nx = {nx} and ny = {ny} => nx . ny = {nx*ny}")
print(f"Matrix size: ({nx*ny} x {nx*ny})")
print(f"Bandwidth: {2*nx}")
print(f"Entries of the factorised matrix: {nx*ny*(2*nx):,}".replace(',',' '))
print(f"Condition number: {eig_val_2d(nx, dx, ny, dy, nx, ny)/eig_val_2d(nx, dx, ny, dy, 1, 1)}")
norm_a = eig_val_2d(nx, dx, ny, dy, nx, ny)
print(f"Norm of A: {norm_a}")

t1 = time.time()    
ulu = spsolve(a.tocsr(), b)
t2 = time.time()
print("\nSolution using a direct method")
print(f"Execution time: {t2-t1}")
res = np.linalg.norm(b - a.dot(ulu))
print(f"||Ax - b|| / ||A|| ||x|| + ||b|| = {res / (norm_a * np.linalg.norm(ulu) + np.linalg.norm(b))}")

## 3d case

$$
\left\{
\begin{aligned}
-\Delta u(x,y,z) & =  f(x,y,z) \quad \text{in} \; \Omega = [0,1] \times [0,1] \times [0,1] \quad \text{with} \; f(x)=1\\
        u(x,y,z) & =  0 \quad \text{on}  \;  \partial \Omega
\end{aligned}
\right.
$$

In [ ]:
nx = 50
ny = 50
nz = 49
dx = 1/(nx+1)
dy = 1/(ny+1)
dz = 1/(nz+1)

# building the sparse matrix
diag = np.repeat(2/dx**2 + 2/dx**2 + 2/dz**2, nx*ny*nz)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny*nz)
diag_y = np.tile(np.repeat([-1/dy**2, 0.], (nx*(ny-1), nz)), nz)
diag_z = np.repeat(-1/dz**2, nx*ny*(nz-1))
a = diags([diag, diag_x, diag_x, diag_y, diag_y, diag_z, diag_z], [0, -1, 1, -nx, nx, -nx*ny, nx*ny])

# right-hand side
b = np.ones(nx*ny*nz)

print(f"3d case: nx = {nx}, ny = {ny} and nz = {nz} => nx . ny . nz = {nx*ny*nz}")
print(f"Matrix size: ({nx*ny*nz} x {nx*ny*nz})")
print(f"Bandwidth: {2*nx*ny}")
print(f"Entries of the factorised matrix: {nx*ny*nz*(2*nx*ny):,}".replace(',',' '))
print(f"Condition number: {eig_val_3d(nx, dx, ny, dy, nz, dz, nx, ny, nz)/eig_val_3d(nx, dx, ny, dy, nz, dz, 1, 1, 1)}")
norm_a = eig_val_3d(nx, dx, ny, dy, nz, dz, nx, ny, nz)
print(f"Norm of A: {norm_a}")

print("\nSolution using a direct method")
t1 = time.time()    
ulu = spsolve(a.tocsr(), b)
t2 = time.time() 
print(f"Execution time: {t2-t1}")
res = np.linalg.norm(b - a.dot(ulu))
print(f"||Ax - b|| / ||A|| ||x|| + ||b|| = {res / (norm_a * np.linalg.norm(ulu) + np.linalg.norm(b))}")

In the three cases proposed in 1d, 2d and 3d, we observe that the backward error is of the order of machine precision, which indicates a great stability of the algorithm used. The number of degrees of freedom used ($\phi = 122500$) is the same, so the size of the matrices involved is the same; the only things that can change are **the fill-in level of the matrix and its condition number**. 

We observe that, **in the 1d case, the condition number of the matrix is very poor (of the order of 6 billion) but that solving with a direct method is very efficient** (of the order of a few hundredths of a second). This situation corresponds to a very low fill-in of the matrix (cf. course slides: fill-in = $3*\phi =367\ 500$). 

The 2d case is **much more favourable in terms of condition number (of the order of 50000) but much less favourable in terms of computing time**: of the order of 2 seconds, that is to say almost one hundred times slower. We see that the fill-in is this time much higher (cf. course slides: fill-in = $(2*\phi^{3/2} \simeq 85\ 750\ 000$, of the order of more than one hundred times larger). 

**The 3d case becomes extremely slow (of the order of 900 seconds, again more than 100 times larger than the 2d case), whereas the condition number is very good**. It is clear that the fill-in is this time very high and makes the algorithm very inefficient (cf. course slides: fill-in = $(2*\phi^{5/3} \simeq 604\ 305\ 866$, of the order of more than 10 times larger than the 2d case and 2000 times larger than the 1d case). 

Overall, the bandwidth of the matrix went from 3 in 1d, to 700 in 2d and 5000 in 3d. **This progressive fill-in makes this kind of approach unusable in 3d for realistic systems with several hundred points in each direction, with a prohibitive storage in terms of memory footprint**. Another strategy must then be found for solving linear systems in 3d: **iterative methods**! 
